In [ ]:
# Finalspark header
import numpy as np
import time
from datetime import datetime, timedelta, timezone
from IPython.display import clear_output
from neuroplatform import StimShape, StimParam , IntanSofware, Trigger, Database, StimPolarity, Experiment

import pandas as pd
from dateutil import parser
from lets_plot import *
from tqdm import tqdm
LetsPlot.setup_html()

In [ ]:
# Establish API credentials
token = "<insert token here>"
exp = Experiment(token)
print(f'Electrodes: {exp.electrodes}') # Electrodes that you can used


In [ ]:
# Attribute list for Stim obj
print(StimParam().__doc__)

## Electrode channel setup

In this specific experiment all 4 organoids are targeted with a series of sequential pulse trains - similiar to the STDP protocol with pre- and post- electrodes, rather here we're attempting to probe the networks ability to form inter-organoid causal pathways.

In [ ]:
# Example electrode setup
first_stim_channel, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
first_stim_channel = StimParam()
first_stim_channel.index = exp.electrodes[1]
first_stim_channel.trigger_key = 0
first_stim_channel.phase_amplitude1 = 10
first_stim_channel.phase_amplitude2 = 10
first_stim_channel.display_attributes()

print(last_updated_paramaters)

In [ ]:
second_stim_channel, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
second_stim_channel = StimParam()
second_stim_channel.index = exp.electrodes[9]
second_stim_channel.trigger_key = 1
second_stim_channel.phase_amplitude1 = 10
second_stim_channel.phase_amplitude2 = 10
second_stim_channel.display_attributes()

print(last_updated_paramaters)

In [ ]:
third_stim_channel, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
third_stim_channel = StimParam()
third_stim_channel.index = exp.electrodes[17]
third_stim_channel.trigger_key = 2
third_stim_channel.phase_amplitude1 = 10
third_stim_channel.phase_amplitude2 = 10
third_stim_channel.display_attributes()

print(last_updated_paramaters)

In [ ]:
fourth_stim_channel, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
fourth_stim_channel = StimParam()
fourth_stim_channel.index = exp.electrodes[25]
fourth_stim_channel.trigger_key = 3
fourth_stim_channel.phase_amplitude1 = 10
fourth_stim_channel.phase_amplitude2 = 10
fourth_stim_channel.display_attributes()

print(last_updated_paramaters)

In [ ]:
# Targeted stimulation frequency - retrieved via initial baseline fft analysis
Hz = 8

sleep = 1/Hz

In [ ]:
# Main block

# See finalspark docs for detailed desc.
intan = IntanSofware()
trigger_gen = Trigger()

# params to send to intan
stim_param_list = [first_stim_channel, second_stim_channel, third_stim_channel, fourth_stim_channel]

# Empty lists for timestamps
preInterStimTimestamps = []
postInterStimTimestamps = []

# Set intan spike-count
intan.set_count([0])

try:
    if exp.start(): # Signal the start of an experiment to all users
        # Measure impedance
        intan.impedance()

        # Disable Variation STD (keep a fixed threshold)
        intan.var_threshold(False)

        # Send stim parameter
        intan.send_stimparam(stim_param_list)
        start_exp = datetime.utcnow()

        # Trigger 0 only = electrode 0
        trigger0 = np.zeros(16, dtype=np.uint8)
        trigger1 = np.zeros(16, dtype=np.uint8)
        trigger2 = np.zeros(16, dtype=np.uint8)
        trigger3 = np.zeros(16, dtype=np.uint8)
        trigger0[0] = 1
        trigger1[0] = 1
        trigger2[0] = 1
        trigger3[0] = 1

        # 10Ms ISI 490Ms Pulse delay STDP loop 60x - Standard Hebbian learning
        for j in range(10):
            current_time = datetime.utcnow()
            preInterStimTimestamps.append(current_time)
            for i in range(60):
                #Electrode 0
                # Pre-  Stim
                trigger_gen.send(trigger0)

                # ISI
                time.sleep(0.01)

                # Post- Stim
                trigger_gen.send(trigger1)


                time.sleep(0.01)

                trigger_gen.send(trigger2)

                time.sleep(0.01)
                trigger_gen.send(trigger3)

                # Pulse delay
                time.sleep(sleep-0.03)
            current_time = datetime.utcnow()
            postInterStimTimestamps.append(current_time)
            time.sleep(50)

        stop_exp = datetime.utcnow()

        # Disable all stims
        for stim in stim_param_list:
            stim.enable = False
        intan.send_stimparam(stim_param_list)

finally:
    trigger_gen.close()
    intan.var_threshold(True)
    intan.close()
    exp.stop()

In [ ]:
# Collect timestamps
print(preInterStimTimestamps)

print(postInterStimTimestamps)

print(start_exp)

print(stop_exp)

In [ ]:
# Load DB libs

import pandas as pd
from dateutil import parser
from lets_plot import *
from tqdm import tqdm
LetsPlot.setup_html()

In [ ]:
# Establish db connection and get exp name
db = Database()
exp_name = exp.exp_name
exp_name

In [ ]:
# Get current time func
def get_current_utc_time():
    """Returns the current UTC time as a datetime object."""
    return datetime.now(timezone.utc)

In [ ]:
# Start time parser - Timestamps gained from experiment protocols
s1 = datetime(2024, 5, 2, 9, 0, 0, 941232)
s1

# s1 = get_current_utc_time() // ALTERNATIVE FORMAT
s1 = parser.parse('2024-06-19T10:35:00.000Z')
s1

# End time parser
s2 = datetime(2024, 5, 2, 12, 0, 0, 956840)
s2

In [ ]:
# Trigger data collection point
triggers = db.get_all_triggers(s1, s2)
triggers

In [ ]:
# SPM data collection point
df_spike_count = db.get_spike_count(s1, s2, exp_name)
df_spike_count

In [ ]:
# Raw spike data collection point
df_spike_event = db.get_spike_event(s1,s2,exp_name)
df_spike_event